# NeuralNode v2.0.2 - Horus Q8 + Replica + Thinking Mode + Telegram

This notebook demonstrates:
- Loading Horus Q8 model
- Replica TTS integration (Female EN-US)
- Interactive chat with voice output
- Thinking mode toggle
- Optional Telegram bot integration

# Install NeuralNode v2.0.2 and all dependencies
!pip install neuralnode==2.0.2

# Install Horus + Replica dependencies
!pip install llama-cpp-python huggingface-hub torch transformers

# Install Replica TTS dependency
!pip install edge-tts>=6.1.0

In [ ]:
# Install NeuralNode v2.0.2
!pip install neuralnode==2.0.2

# Install Horus extras for GGUF support
!pip install llama-cpp-python huggingface-hub torch transformers

## Step 2: Import Libraries

In [ ]:
import neuralnode as nn
import os
import time
from IPython.display import Audio, display, clear_output

## Step 3: Configuration

In [ ]:
# Configuration
MODEL_ID = "tokenaii/Hours-1.0-4B-GGUF/Horus-1.0-4B-Q8_0.gguf"  # Q8 version
CACHE_DIR = "/content/neuralnode_models"  # Colab path
REPLICA_VOICE = "replic-aria-language{en-us}"  # Female EN-US
TELEGRAM_TOKEN = "REDACTED_TELEGRAM_TOKEN"

# Options
ENABLE_THINKING = True  # Toggle thinking mode
ENABLE_TELEGRAM = False  # Toggle Telegram bot

print("✅ Configuration loaded")
print(f"Model: {MODEL_ID}")
print(f"Voice: {REPLICA_VOICE}")
print(f"Thinking Mode: {ENABLE_THINKING}")
print(f"Telegram Bot: {ENABLE_TELEGRAM}")

## Step 4: Load Horus Q8 Model

In [ ]:
# Create cache directory
os.makedirs(CACHE_DIR, exist_ok=True)

# Load Horus model with all features
print("⏳ Loading Horus Q8 model...")
print("This may take 5-10 minutes on first download...")

model = nn.HorusModel(
    model_id=MODEL_ID,
    device="cpu",
    cache_dir=CACHE_DIR,
    n_ctx=8192,  # Original context window
    verbose=False
).load()

print("✅ Model loaded successfully!")
print(f"Context Window: 8192 tokens")
print(f"Chat Template: horus_unified")

## Step 5: Setup Replica TTS

In [ ]:
# Initialize Replica TTS
tts = nn.ReplicaTTS(voice_id=REPLICA_VOICE)

# Test voice
print("🔊 Testing Replica TTS...")
tts.save_to_file("Hello! I'm ready to chat with you.", "/content/test_voice.mp3")
display(Audio("/content/test_voice.mp3", autoplay=True))
print("✅ Voice test complete!")

## Step 6: Setup Thinking Mode (Optional)

In [ ]:
if ENABLE_THINKING:
    # Wrap model with thinking mode
    thinker = nn.ThinkingMode(
        model,
        steps=["understand", "plan", "execute", "verify"],
        show_thinking=True,
        concise=True,
        max_tokens_per_step=100,
        max_final_tokens=200
    )
    chat_model = thinker
    print("🧠 Thinking Mode: ENABLED")
    print("Thinking will be shown between <think> tags")
else:
    chat_model = model
    print("🧠 Thinking Mode: DISABLED")

## Step 7: Chat Function with Voice Output

In [ ]:
def chat_with_voice(user_message, show_thinking=ENABLE_THINKING):
    """Chat with Horus and get voice response"""
    
    print(f"\n👤 You: {user_message}")
    print("🤖 Horus is thinking...")
    
    if show_thinking and ENABLE_THINKING:
        # Get response with thinking
        response = chat_model.chat(user_message)
        
        # Display thinking
        if response.thinking:
            print("\n🧠 <think>")
            print(response.thinking)
            print("</think>")
        
        text_response = response.content
    else:
        # Normal chat
        messages = [{"role": "user", "content": user_message}]
        response = model.chat(messages, max_new_tokens=200)
        text_response = response.content
    
    # Generate audio
    audio_file = f"/content/response_{int(time.time())}.mp3"
    tts.save_to_file(text_response, audio_file)
    
    # Play audio
    print("\n🔊 Audio Response:")
    display(Audio(audio_file, autoplay=True))
    
    # Display text
    print("\n📝 Text Response:")
    print(f"🤖 Horus: {text_response}")
    
    return text_response

print("✅ Chat function ready!")

## Step 8: Interactive Chat (Try it!)

In [ ]:
# Example 1: Simple greeting
chat_with_voice("Hello! Introduce yourself briefly.")

In [ ]:
# Example 2: Ask a question
chat_with_voice("What is 15 times 23?")

In [ ]:
# Example 3: Complex question (shows thinking if enabled)
chat_with_voice("Explain how photosynthesis works in simple terms.")

## Step 9: Custom Chat (Enter your own message)

In [ ]:
# Type your own message here
your_message = "Tell me a short joke"  # Change this to your message

chat_with_voice(your_message)

## Step 10: Telegram Bot Setup (Optional)

In [ ]:
if ENABLE_TELEGRAM:
    print("🤖 Setting up Telegram Bot...")
    
    # Create NeuralNode instance
    ai = nn.NeuralNode(
        provider="horus",
        model=MODEL_ID,
        model_settings={"cache_dir": CACHE_DIR}
    )
    
    # Create agent with thinking
    agent = ai.agent(
        agent_type="simple",
        skills=["datetime"],
        thinking=ENABLE_THINKING
    )
    
    # Configure bot
    bot_config = nn.TelegramBotConfig(
        token=TELEGRAM_TOKEN,
        enable_voice=True,
        enable_documents=True,
        reply_mode="both",  # text + voice
        voice_reply_voice_id=REPLICA_VOICE,
    )
    
    # Create bot
    bot = nn.TelegramBot(
        token=TELEGRAM_TOKEN,
        agent=agent,
        config=bot_config,
    )
    
    print("✅ Telegram bot configured!")
    print("\n🚀 To start the bot, run:")
    print("   bot.start()")
    print("\n⚠️  Note: This will block the notebook until stopped")
    print("   Press Ctrl+C to stop")
    
    # Ask before starting
    start = input("\nStart Telegram bot now? (y/n): ").lower().strip()
    if start == "y":
        print("\n🟢 Starting Telegram bot...")
        print("You can now chat with Horus on Telegram!")
        try:
            bot.start()
        except KeyboardInterrupt:
            print("\n🛑 Bot stopped by user")
else:
    print("⏸️  Telegram bot disabled. Set ENABLE_TELEGRAM = True to enable.")

## Summary & Quick Reference

In [ ]:
# Quick reference
print("═" * 60)
print("NeuralNode v2.0.2 - Horus + Replica Demo")
print("═" * 60)
print("\n✅ Features Ready:")
print(f"   • Model: Horus Q8 (8192 context)")
print(f"   • Voice: Replica {REPLICA_VOICE}")
print(f"   • Thinking: {'ON' if ENABLE_THINKING else 'OFF'}")
print(f"   • Telegram: {'ON' if ENABLE_TELEGRAM else 'OFF'}")
print("\n📌 Quick Commands:")
print("   chat_with_voice('Your message')")
print("\n📝 Notes:")
print("   • Change ENABLE_THINKING to toggle thinking mode")
print("   • Change ENABLE_TELEGRAM to toggle Telegram bot")
print("   • Voice responses auto-play")
print("   • Text responses shown below audio")
print("═" * 60)